# India VAHAN Data Cleaning & Validation Pipeline

Reads the raw scraped VAHAN CSVs and supporting data (GSDP, population,
policies, charging stations) from S3, cleans and merges them, and writes
the result back to S3.

This is a more defensive version of the original cleaning notebook: every
transformation step is paired with a validation check that surfaces
problems instead of silently swallowing them — unmapped state names,
unmapped vehicle classes, values that fail numeric parsing, row-count
drift across merges, and duplicate-grain checks on the final table. The
goal is that if something in the raw data changes shape (a new vehicle
class appears, VAHAN adds a state, a column gets renamed upstream), this
notebook tells you loudly rather than quietly producing a slightly wrong
`final_df`.

In [ ]:
import io
import os

import boto3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.max_columns", None)

BUCKET = "vahan-project-raw-486491621202-ap-south-1-an"
RAW_PREFIX = "scraped/"
SUPPORTING_PREFIX = "supporting_data/"
OUTPUT_PREFIX = "processed/"

s3 = boto3.client("s3")


def list_keys(bucket, prefix):
    keys = []
    paginator = s3.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            if not obj["Key"].endswith("/"):
                keys.append(obj["Key"])
    return keys


def read_csv_from_s3(bucket, key, **kwargs):
    obj = s3.get_object(Bucket=bucket, Key=key)
    return pd.read_csv(io.BytesIO(obj["Body"].read()), **kwargs)


def read_excel_from_s3(bucket, key, **kwargs):
    obj = s3.get_object(Bucket=bucket, Key=key)
    return pd.read_excel(io.BytesIO(obj["Body"].read()), **kwargs)


def write_csv_to_s3(df, bucket, key):
    buf = io.StringIO()
    df.to_csv(buf, index=False)
    s3.put_object(Bucket=bucket, Key=key, Body=buf.getvalue())
    print(f"  Wrote s3://{bucket}/{key} ({len(df):,} rows)")


def check(condition: bool, ok_msg: str, fail_msg: str, raise_on_fail: bool = False):
    """Lightweight inline assertion that prints instead of just crashing,
    so a validation failure is visible in the notebook's output history
    even if you don't re-run from scratch."""
    if condition:
        print(f"  [OK] {ok_msg}")
    else:
        print(f"  [CHECK FAILED] {fail_msg}")
        if raise_on_fail:
            raise AssertionError(fail_msg)

## 1. Static lookup tables

`MONTH_MAP`, `VEHICLE_CAT` / `VEHICLE_LOOKUP`, and `STATE_MAPPING` are
copied verbatim from `clean_data_s3.py` (not retyped), so this notebook
and the production script can never silently drift apart on the actual
mapping logic.

In [ ]:
MONTH_MAP = {
    'JAN': 1, 'FEB': 2, 'MAR': 3, 'APR': 4, 'MAY': 5, 'JUN': 6,
    'JUL': 7, 'AUG': 8, 'SEP': 9, 'OCT': 10, 'NOV': 11, 'DEC': 12,
}


VEHICLE_CAT = {
    'Agricultural Vehicle': [
        'AGRICULTURAL TRACTOR', 'POWER TILLER', 'HARVESTER',
        'TRAILER (AGRICULTURAL)', 'POWER TILLER (COMMERCIAL)',
        'TRACTOR (COMMERCIAL)', 'PULLER TRACTOR'
    ],
    'Bus': [
        'OMNI BUS (PRIVATE USE)', 'BUS', 'SCHOOL BUS',
        'EDUCATIONAL INSTITUTION BUS', 'OMNI BUS'
    ],
    'Car': ['MOTOR CAR'],
    'Construction & Industrial Equipment': [
        'FORK LIFT', 'CRANE MOUNTED VEHICLE', 'CONSTRUCTION EQUIPMENT VEHICLE',
        'ROAD ROLLER', 'EXCAVATOR (NT)', 'BULLDOZER',
        'EARTH MOVING EQUIPMENT', 'EXCAVATOR (COMMERCIAL)',
        'CONSTRUCTION EQUIPMENT VEHICLE (COMMERCIAL)'
    ],
    'Emergency Vehicle': [
        'AMBULANCE', 'ANIMAL AMBULANCE', 'FIRE TENDERS',
        'SNORKED LADDERS', 'FIRE FIGHTING VEHICLE', 'HEARSES'
    ],
    'Goods Vehicle': [
        'GOODS CARRIER', 'AUXILIARY TRAILER', 'ARTICULATED VEHICLE',
        'DUMPER', 'TRAILER (COMMERCIAL)', 'TRACTOR-TROLLEY(COMMERCIAL)',
        'SEMI-TRAILER (COMMERCIAL)', 'MODULAR HYDRAULIC TRAILER'
    ],
    'Quadricycle': ['QUADRICYCLE (PRIVATE)', 'QUADRICYCLE (COMMERCIAL)'],
    'Recreational Vehicle': [
        'CAMPER VAN / TRAILER (PRIVATE USE)', 'TRAILER FOR PERSONAL USE',
        'MOTOR CARAVAN', 'CAMPER VAN / TRAILER'
    ],
    'Service Vehicle': [
        'PRIVATE SERVICE VEHICLE (INDIVIDUAL USE)', 'PRIVATE SERVICE VEHICLE'
    ],
    'Special Purpose Vehicle': [
        'VEHICLE FITTED WITH RIG', 'VEHICLE FITTED WITH GENERATOR',
        'VEHICLE FITTED WITH COMPRESSOR', 'TOW TRUCK', 'BREAKDOWN VAN',
        'RECOVERY VEHICLE', 'TOWER WAGON', 'TREE TRIMMING VEHICLE',
        'ARMOURED/SPECIALISED VEHICLE', 'MOBILE WORKSHOP', 'CASH VAN',
        'ADAPTED VEHICLE', 'MOBILE CLINIC', 'X-RAY VAN', 'LIBRARY VAN',
        'MOBILE CANTEEN'
    ],
    'Taxi / Cab': ['LUXURY CAB', 'MAXI CAB', 'MOTOR CAB'],
    'Three Wheeler': [
        'E-RICKSHAW WITH CART (G)', 'THREE WHEELER (GOODS)',
        'THREE WHEELER (PERSONAL)', 'E-RICKSHAW(P)',
        'THREE WHEELER (PASSENGER)'
    ],
    'Two Wheeler': [
        'MOTOR CYCLE/SCOOTER-SIDECAR(T)', 'MOTOR CYCLE/SCOOTER-WITH TRAILER',
        'M-CYCLE/SCOOTER', 'M-CYCLE/SCOOTER-WITH SIDE CAR', 'MOPED',
        'MOTORISED CYCLE (CC > 25CC)', 'MOTOR CYCLE/SCOOTER-USED FOR HIRE'
    ],
    'Vintage Vehicle': ['VINTAGE MOTOR VEHICLE'],
}
VEHICLE_LOOKUP = {
    vehicle: category
    for category, vehicles in VEHICLE_CAT.items()
    for vehicle in vehicles
}


STATE_MAPPING = {
    'All States': 'All States',
    'Andaman & Nicobar (UT)': 'Andaman and Nicobar Islands',
    'Andaman & Nicobar Island': 'Andaman and Nicobar Islands',
    'Andaman & Nicobar Islands': 'Andaman and Nicobar Islands',
    'Andaman And Nicobar Islands': 'Andaman and Nicobar Islands',
    'Andhra Pradesh': 'Andhra Pradesh',
    'Arunachal Pradesh': 'Arunachal Pradesh',
    'Assam': 'Assam',
    'Bihar': 'Bihar',
    'Chandigarh': 'Chandigarh',
    'Chandigarh (UT)': 'Chandigarh',
    'Chhattisgarh': 'Chattisgarh',
    'DNH and DD (UT)': 'Dadra & Nagar Haveli and Daman & Diu',
    'Dadra and Nagar Haveli and Daman and Diu': 'Dadra & Nagar Haveli and Daman & Diu',
    'Delhi': 'Delhi',
    'Goa': 'Goa',
    'Gujarat': 'Gujarat',
    'Haryana': 'Haryana',
    'Himachal Pradesh': 'Himachal Pradesh',
    'Jammu & Kashmir': 'Jammu and Kashmir',
    'Jammu & Kashmir*': 'Jammu and Kashmir',
    'Jammu And Kashmir': 'Jammu and Kashmir',
    'Jammu and Kashmir': 'Jammu and Kashmir',
    'Jharkhand': 'Jharkhand',
    'Karnataka': 'Karnataka',
    'Kerala': 'Kerala',
    'Ladakh': 'Ladakh',
    'Ladakh (UT)': 'Ladakh',
    'Lakshadweep': 'Lakshadweep Islands',
    'Lakshadweep (UT)': 'Lakshadweep Islands',
    'Madhya Pradesh': 'Madhya Pradesh',
    'Maharashtra': 'Maharashtra',
    'Manipur': 'Manipur',
    'Meghalaya': 'Meghalaya',
    'Mizoram': 'Mizoram',
    'NCT of Delhi': 'Delhi',
    'Nagaland': 'Nagaland',
    'Odisha': 'Odisha',
    'Puducherry': 'Pondicherry',
    'Puducherry (UT)': 'Pondicherry',
    'Punjab': 'Punjab',
    'Rajasthan': 'Rajasthan',
    'Sikkim': 'Sikkim',
    'Tamil Nadu': 'Tamil Nadu',
    'Telangana': 'Telangana',
    'Tripura': 'Tripura',
    'UT of DNH and DD': 'Dadra & Nagar Haveli and Daman & Diu',
    'Uttar Pradesh': 'Uttar Pradesh',
    'Uttarakhand': 'Uttarakhand',
    'West Bengal': 'West Bengal',
}


In [ ]:
print(f"MONTH_MAP: {len(MONTH_MAP)} entries")
print(f"VEHICLE_LOOKUP: {len(VEHICLE_LOOKUP)} vehicle classes mapped across {len(VEHICLE_CAT)} categories")
print(f"STATE_MAPPING: {len(STATE_MAPPING)} raw state-name variants mapped")

## 2. Load raw VAHAN files from S3

Before combining anything, look at what's actually there — file count,
row count per file, and column consistency. A silent `--out-file` typo or
a partially-uploaded file should show up here, not three steps later as a
mysterious row-count discrepancy.

In [ ]:
keys = list_keys(BUCKET, RAW_PREFIX)
check(len(keys) > 0, f"Found {len(keys)} raw file(s) under s3://{BUCKET}/{RAW_PREFIX}",
      f"No files found under s3://{BUCKET}/{RAW_PREFIX} — check the prefix / bucket name.",
      raise_on_fail=True)

file_summary = []
raw_dfs = {}
for key in keys:
    fuel_name = os.path.splitext(os.path.basename(key))[0]
    df = read_csv_from_s3(BUCKET, key)
    raw_dfs[fuel_name] = df
    file_summary.append({"fuel_type": fuel_name, "key": key, "rows": len(df), "columns": len(df.columns)})

file_summary_df = pd.DataFrame(file_summary).sort_values("fuel_type")
file_summary_df

### 2.1 Column consistency check

All files should share identical columns before concatenation. If one
file's columns differ (a scrape run with a different `--yaxis`, say), it
needs to be reindexed rather than silently misaligned during `concat`.

In [ ]:
expected_columns = list(next(iter(raw_dfs.values())).columns)
mismatches = {name: list(df.columns) for name, df in raw_dfs.items() if list(df.columns) != expected_columns}

check(len(mismatches) == 0,
      f"All {len(raw_dfs)} files share identical columns: {expected_columns}",
      f"{len(mismatches)} file(s) have mismatched columns and will be reindexed: {list(mismatches.keys())}")

if mismatches:
    for name, cols in mismatches.items():
        print(f"    {name}: {cols}")

In [ ]:
df_list = []
for name, df in raw_dfs.items():
    if list(df.columns) != expected_columns:
        df = df.reindex(columns=expected_columns)
    df = df.copy()
    df['fuel_type'] = name
    df_list.append(df)

combined_df = pd.concat(df_list, ignore_index=True)
print(f"Combined shape: {combined_df.shape}")

expected_total_rows = file_summary_df["rows"].sum()
check(len(combined_df) == expected_total_rows,
      f"Combined row count ({len(combined_df):,}) matches sum of individual files ({expected_total_rows:,})",
      f"Row count mismatch: combined={len(combined_df):,} vs sum of files={expected_total_rows:,} — rows were lost or duplicated during concat.",
      raise_on_fail=True)

## 3. Raw data quality before cleaning

A quick look at nulls and duplicates in the raw combined data, before any
transformation. This is the baseline to compare against after cleaning —
if cleaning *introduces* nulls somewhere it shouldn't, this is how you'd
notice.

In [ ]:
raw_nulls = combined_df.isna().sum()
raw_nulls = raw_nulls[raw_nulls > 0]
if len(raw_nulls):
    print("Columns with nulls in raw data:")
    print(raw_nulls)
else:
    print("No nulls in raw combined data.")

raw_dupes = combined_df.duplicated().sum()
print(f"\nExact duplicate rows in raw data: {raw_dupes}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(combined_df.isna(), cbar=False, yticklabels=False, cmap="rocket_r", ax=ax)
ax.set_title("Missingness map — raw combined data (dark = missing)")
plt.tight_layout()
plt.show()

## 4. Clean the core VAHAN columns

Select relevant columns, coerce `value` to numeric, map vehicle class →
category, map month name → number, derive financial year.

In [ ]:
df = combined_df.copy()
df.columns = df.columns.str.lower().str.replace(' ', '_')
df = df[['year', 'month_wise', 'state', 'vehicle_class', 'fuel_type', 'value']]
df.head()

### 4.1 Numeric parsing of `value` — check what fails

`value` arrives as text (commas, possible stray whitespace). Coercing with
`errors='coerce'` will silently turn anything unparseable into `NaN` —
worth seeing exactly which raw strings that happens to, rather than just
trusting the count.

In [ ]:
raw_value = df['value'].astype(str).str.replace(',', '').str.strip()
parsed_value = pd.to_numeric(raw_value, errors='coerce')

failed_parse_mask = parsed_value.isna() & df['value'].notna()
n_failed = failed_parse_mask.sum()

check(n_failed == 0,
      "Every non-null 'value' parsed to numeric cleanly.",
      f"{n_failed} row(s) had a 'value' that failed numeric parsing — sample of raw strings below.")

if n_failed:
    print(df.loc[failed_parse_mask, 'value'].value_counts().head(10))

df['value'] = parsed_value

### 4.2 Vehicle class mapping — check for unmapped classes

If VAHAN introduces a new vehicle class not in `VEHICLE_LOOKUP`, those
rows would silently get `vehicle_category = NaN`. This check surfaces
exactly which raw `vehicle_class` values aren't covered, and how many
rows/registrations they represent — small stray categories are a minor
gap, a large one means the lookup table needs an update before trusting
`vehicle_category`-based analysis.

In [ ]:
df['vehicle_category'] = df['vehicle_class'].map(VEHICLE_LOOKUP)

unmapped_mask = df['vehicle_category'].isna() & df['vehicle_class'].notna()
unmapped_classes = df.loc[unmapped_mask, 'vehicle_class'].value_counts()

check(len(unmapped_classes) == 0,
      "Every vehicle_class in the data is covered by VEHICLE_LOOKUP.",
      f"{len(unmapped_classes)} distinct vehicle_class value(s) are NOT in VEHICLE_LOOKUP "
      f"({unmapped_mask.sum():,} rows, {100*unmapped_mask.sum()/len(df):.2f}% of data):")

if len(unmapped_classes):
    print(unmapped_classes)

### 4.3 Month mapping — check for unmapped months

In [ ]:
df['month_number'] = df['month_wise'].map(MONTH_MAP)

unmapped_month_mask = df['month_number'].isna() & df['month_wise'].notna()
check(unmapped_month_mask.sum() == 0,
      "Every month_wise value mapped to a month number.",
      f"{unmapped_month_mask.sum()} row(s) had an unmapped month_wise value: "
      f"{df.loc[unmapped_month_mask, 'month_wise'].unique().tolist()}")

### 4.4 Financial year derivation

India's financial year runs April–March. Month >= 4 belongs to
`year-(year+1)`; month < 4 belongs to `(year-1)-year`.

In [ ]:
df["financial_year"] = np.where(
    df["month_number"] >= 4,
    df["year"].astype(str) + "-" + (df["year"] + 1).astype(str).str[2:],
    (df["year"] - 1).astype(str) + "-" + df["year"].astype(str).str[2:],
)

check(df["financial_year"].isna().sum() == 0,
      "Every row has a derived financial_year.",
      f"{df['financial_year'].isna().sum()} row(s) have a null financial_year "
      "(likely from a null month_number upstream).")

df[["year", "month_wise", "month_number", "financial_year"]].drop_duplicates().sort_values(["year", "month_number"]).head(10)

### 4.5 State name reconciliation — check for unmapped states

The most fragile step: raw state names vary across sources
(`'Jammu & Kashmir*'` vs `'Jammu And Kashmir'` vs `'Jammu and Kashmir'`,
etc.), and `STATE_MAPPING` was built by manually reviewing a rapidfuzz
pass over the sources available at the time. If VAHAN adds a state/UT or
spells one differently in a future scrape, it would silently become
`NaN` here without this check.

In [ ]:
raw_states = df['state'].unique()
unmapped_states = sorted(set(raw_states) - set(STATE_MAPPING.keys()))

check(len(unmapped_states) == 0,
      f"All {len(raw_states)} raw state name variants are covered by STATE_MAPPING.",
      f"{len(unmapped_states)} raw state name(s) are NOT in STATE_MAPPING and will become null: {unmapped_states}")

df['state'] = df['state'].map(STATE_MAPPING)

null_state_rows = df['state'].isna().sum()
check(null_state_rows == 0,
      "No rows lost their state after mapping.",
      f"{null_state_rows} row(s) ({100*null_state_rows/len(df):.2f}%) have a null state after mapping — "
      "these will fail to join to dim_state downstream.")

## 5. Supporting data — GSDP, population, policies, charging stations

Loaded from `supporting_data/` in S3. Each gets the same state-name
mapping applied, with the same kind of unmapped-state check.

In [ ]:
gsdp = read_csv_from_s3(BUCKET, f"{SUPPORTING_PREFIX}gsdp.csv")
gsdp_melted = gsdp.melt(id_vars="State/Union Territory", var_name="financial_year", value_name="gsdp_lakhs")
gsdp_melted['gsdp_lakhs'] = pd.to_numeric(
    gsdp_melted['gsdp_lakhs'].astype(str).str.replace(',', '').str.strip(), errors='coerce'
).astype("Int64")
gsdp_melted.columns = gsdp_melted.columns.str.replace('/', '_').str.replace(' ', '_')
gsdp_melted = gsdp_melted.rename(columns={'State_Union_Territory': 'state'})

unmapped_gsdp_states = sorted(set(gsdp_melted['state'].unique()) - set(STATE_MAPPING.keys()))
check(len(unmapped_gsdp_states) == 0,
      "All GSDP state names are covered by STATE_MAPPING.",
      f"Unmapped GSDP state names: {unmapped_gsdp_states}")

gsdp_melted['state'] = gsdp_melted['state'].map(STATE_MAPPING)
gsdp_melted.head()

In [ ]:
dfs = read_excel_from_s3(BUCKET, f"{SUPPORTING_PREFIX}support.xlsx", sheet_name=None)
print(f"Sheets found: {list(dfs.keys())}")

expected_sheets = {"population", "policies", "charging_stations", "state_codes"}
check(expected_sheets.issubset(dfs.keys()),
      "All expected sheets are present.",
      f"Missing expected sheet(s): {expected_sheets - set(dfs.keys())}")

population = dfs['population']
policies = dfs['policies']
charging_stations = dfs['charging_stations']
state_codes = dfs['state_codes']

In [ ]:
population = population.rename(columns={'State/Union Territory': 'State'})

unmapped_pop_states = sorted(set(population['State'].unique()) - set(STATE_MAPPING.keys()))
check(len(unmapped_pop_states) == 0,
      "All population state names are covered by STATE_MAPPING.",
      f"Unmapped population state names: {unmapped_pop_states}")

population['State'] = population['State'].map(STATE_MAPPING)
population_melted = population.melt(id_vars="State", var_name="year", value_name="population")
population_melted['year'] = pd.to_numeric(population_melted['year']).astype("int64")
population_melted.head()

In [ ]:
for name, sheet_df in [("policies", policies), ("charging_stations", charging_stations)]:
    unmapped = sorted(set(sheet_df['State'].unique()) - set(STATE_MAPPING.keys()))
    check(len(unmapped) == 0,
          f"All {name} state names are covered by STATE_MAPPING.",
          f"Unmapped {name} state names: {unmapped}")
    sheet_df['State'] = sheet_df['State'].map(STATE_MAPPING)

policies.head()

In [ ]:
charging_stations.head()

Note: `policies` and `charging_stations` are cleaned (state names
normalized) but deliberately **not merged** into `final_df` — they don't
share a clean join grain with the registrations fact table (see the
schema design discussion). They're exported separately below for the
`policy_events` and `fact_charging_stations` tables instead.

## 6. Final merge

`vahan` (cleaned registrations) left-joined with `state_codes`, then
`population`, then `gsdp` — all on `state` (+ `year`/`financial_year`
where relevant). A left join should never change the row count from the
left-hand table; that's checked explicitly at each step rather than
assumed.

In [ ]:
vahan = df  # the cleaned registrations dataframe from section 4
pre_merge_rows = len(vahan)

final_df = vahan.merge(state_codes, how='left', left_on='state', right_on='State').drop(columns='State')
check(len(final_df) == pre_merge_rows,
      "Row count unchanged after merging state_codes.",
      f"Row count changed after state_codes merge: {pre_merge_rows:,} -> {len(final_df):,} "
      "(state_codes likely has duplicate state entries).",
      raise_on_fail=True)

final_df = final_df.merge(population_melted, how='left', left_on=['state', 'year'], right_on=['State', 'year']).drop(columns='State')
check(len(final_df) == pre_merge_rows,
      "Row count unchanged after merging population.",
      f"Row count changed after population merge: {pre_merge_rows:,} -> {len(final_df):,}.",
      raise_on_fail=True)

final_df = final_df.merge(gsdp_melted, how='left', on=['state', 'financial_year'])
check(len(final_df) == pre_merge_rows,
      "Row count unchanged after merging GSDP.",
      f"Row count changed after GSDP merge: {pre_merge_rows:,} -> {len(final_df):,}.",
      raise_on_fail=True)

print(f"final_df shape: {final_df.shape}")

In [ ]:
final_df.columns = (
    final_df.columns.str.replace('/', '_')
    .str.replace(' ', '_')
    .str.lower()
    .str.replace('tin', 'state_id')
    .str.replace('month_wise', 'month')
    .str.replace('value', 'registrations')
)

final_df = final_df[[
    'year', 'month_number', 'month', 'financial_year', 'state_id', 'state_code',
    'state', 'population', 'gsdp_lakhs', 'fuel_type', 'vehicle_class',
    'vehicle_category', 'registrations',
]]

final_df.head()

## 7. Final validation checks

Before trusting `final_df` enough to load it into RDS: check the grain is
actually unique (no duplicate state+year+month+fuel+vehicle_class rows),
look at where nulls remain and whether they're expected, and check for
any registrations values that shouldn't be possible.

In [ ]:
grain_cols = ['state', 'year', 'month_number', 'fuel_type', 'vehicle_class']
dupe_grain = final_df.duplicated(subset=grain_cols).sum()
check(dupe_grain == 0,
      "final_df's grain (state, year, month, fuel_type, vehicle_class) is unique — no duplicate rows.",
      f"{dupe_grain} duplicate-grain row(s) found — these would double-count in any aggregation.")

check((final_df['registrations'] >= 0).all(),
      "No negative registrations values.",
      f"{(final_df['registrations'] < 0).sum()} row(s) have negative registrations — investigate the source file.")

In [ ]:
null_summary = final_df.isna().sum()
null_pct = (100 * null_summary / len(final_df)).round(2)
null_report = pd.DataFrame({"null_count": null_summary, "null_pct": null_pct})
null_report[null_report["null_count"] > 0]

Expected nulls here: `gsdp_lakhs` will show real nulls for the most
recent financial years, since GSDP figures are released with a lag —
that's a genuine data gap, not a pipeline bug. Any nulls in `state_id`,
`state_code`, `vehicle_category`, or `population` are worth investigating
via the checks in sections 4–6 above rather than accepted at face
value.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

sns.histplot(final_df['registrations'].clip(lower=1), bins=50, log_scale=True, ax=axes[0])
axes[0].set_title("Distribution of registrations (log scale)")
axes[0].set_xlabel("Registrations (log scale)")

final_df.isna().mean().mul(100).sort_values(ascending=False).head(10).plot.barh(ax=axes[1], color="firebrick")
axes[1].set_title("Top columns by % null")
axes[1].set_xlabel("% null")

plt.tight_layout()
plt.show()

## 8. Write cleaned outputs to S3

In [ ]:
write_csv_to_s3(final_df, BUCKET, f"{OUTPUT_PREFIX}final_df.csv")
write_csv_to_s3(policies, BUCKET, f"{OUTPUT_PREFIX}policies.csv")
write_csv_to_s3(charging_stations, BUCKET, f"{OUTPUT_PREFIX}charging_stations.csv")

print("\nDone.")

## Data quality notes

A place to record anything the checks above actually flagged when you ran
this against the real data — not hypothetical issues, what you *actually
saw* this run:

- Any unmapped states, vehicle classes, or months (sections 4.2–4.5)?
- What fraction of `gsdp_lakhs` is null, and for which financial years —
  does that match what you'd expect from GSDP's known reporting lag?
- Any duplicate-grain rows in the final check (section 7)?

This becomes the "Data Quality Notes" section of your README — reviewers
weigh evidence that you checked your data over an unqualified "the data
is clean."
